In [1]:
import json
import zipfile
from pathlib import Path

def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

def zip_results(results_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(results_path, arcname="results.json")

def article_to_doc(article):
    parts = article.split("|")
    if len(parts) >= 2:
        return "|".join(parts[:2])
    return article

def unique_keep_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

def filter_by_score_and_top_k(data, score_threshold=0.0, top_k=-1, keep_scores=False):
    for item in data:
        score_items = item.get("scores", [])
        filtered = [
            x for x in score_items
            if x.get("score", 0.0) >= score_threshold
        ]
        if top_k != -1:
            filtered = filtered[:top_k]
        filtered_articles = [x["article"] for x in filtered]
        filtered_docs = unique_keep_order([
            article_to_doc(article)
            for article in filtered_articles
        ])
        item["relevant_articles"] = filtered_articles
        item["relevant_docs"] = filtered_docs
        if keep_scores:
            item["scores"] = filtered
        else:
            item.pop("scores", None)
    return data

def count_articles_per_question(data, verbose=True):
    """
    Đếm số lượng relevant_articles của mỗi question.
    Trả về dict: {question_id: count}
    In thống kê tổng quan nếu verbose=True.
    """
    counts = {}
    for item in data:
        qid = item.get("question_id", item.get("id", "unknown"))
        n = len(item.get("relevant_articles", []))
        counts[qid] = n

    if verbose:
        values = list(counts.values())
        total = len(values)
        print(f"Tổng số questions : {total}")
        print(f"Tổng số articles  : {sum(values)}")
        print(f"Trung bình / question: {sum(values)/total:.2f}" if total else "N/A")
        print(f"Max               : {max(values)} articles")
        print(f"Min               : {min(values)} articles")
        zero = sum(1 for v in values if v == 0)
        print(f"Questions không có article nào: {zero}")

    return counts


# ── Main ──────────────────────────────────────────────────────────────────────
input_path = "/kaggle/input/datasets/vydtruynthng/r2ai-ai-team-data/bge_m3_biencoder_threshold_submission.json"
output_dir = Path("bge")
score_threshold = 0.90
top_k = -1
keep_scores = False

output_dir.mkdir(parents=True, exist_ok=True)
data = load_json(input_path)
print(data[0].keys())

data = filter_by_score_and_top_k(
    data=data,
    score_threshold=score_threshold,
    top_k=top_k,
    keep_scores=keep_scores,
)

counts = count_articles_per_question(data, verbose=True)

results_path = output_dir / "results.json"
zip_path = output_dir / "submission.zip"
save_json(data, results_path)
zip_results(results_path, zip_path)
print(f"Wrote {results_path}")
print(f"Wrote {zip_path}")

dict_keys(['id', 'question', 'answer', 'relevant_docs', 'relevant_articles', 'scores'])
Tổng số questions : 2000
Tổng số articles  : 8615
Trung bình / question: 4.31
Max               : 40 articles
Min               : 1 articles
Questions không có article nào: 0
Wrote bge/results.json
Wrote bge/submission.zip
